# Notebook 3: Testing Modified SGP4 on LAGEOS-1


Note that the positional and velocity outputs below are in the True Equator Mean Equinox (TEME) reference frame according to [SGP4 PyPI documentation](https://pypi.org/project/sgp4/), and these are handled internally by Brent for TEME-GCRF conversion.

In [ ]:
# Imports and paths
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from contextlib import redirect_stderr

# Prevent path errors:
os.environ["JAVA_HOME"] = str(Path(sys.prefix) / "Library" / "jre")

from brent import Constants
from brent.propagators.analytical.tle import (
    TLEPropagator,
    TLEPropagatorMethod
)

# Paths:
repo_path = Path.cwd().resolve()
if repo_path.name == "experiments":
    repo_path = repo_path.parent

data_path = repo_path / "data"  # data folder
download_path = repo_path / "download"  # downloads folder for TLEs/SP3

# Input files for TLEs + SP3:
tle_path = download_path / "tle" / "8820.json"
sp3_path = data_path / "states" / "lageos1_sp3_2025.csv"

# Output paths:
comparison_path = data_path / "sgp4mods-comparison"  # writing data files
comparison_path.mkdir(parents=True, exist_ok=True)

ImportError: DLL load failed while importing pythalassa: The specified module could not be found.

## Obtaining Satellite States for Baseline \& Modified SGP4 Implementation (nominal $C_X$):

I first compare the unmodified SGP4 implementation in the branch `sgp4-baseline` with my current branch `prelim-experiments` with the modified implementation (but nominal $C_X$ that has not yet been changed). This will be the final test before doing the changes to $C_X$.

The firsrt step is to make sure the same epochs used in [Notebook 1](01_tle_sp3_residuals.ipynb) with 8,328 epochs across January 2025 to December 2025 will be used again for an annual comparison, to compare satellite states $X_{\text{baseline}}$ and $X_{\text{Hujsak}}$.

In [2]:
# Initialisations:

# Identify current SGP4 implementation
branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo_path,
    text=True
).strip()

filenames = {
    "sgp4-baseline": "lageos1_sgp4_baseline_2025.csv",
    "prelim-experiments": "lageos1_hujsak_nominal_cx_2025.csv",
}

if branch not in filenames:
    raise ValueError(f"Unexpected branch: {branch}")

output_file = comparison_path / filenames[branch]

# Load existing comparison epochs from Notebook 1:
sp3_df = pd.read_csv(sp3_path, parse_dates=["epoch_utc"])
dates = sp3_df["epoch_utc"]

assert len(dates) == 8328
assert dates.is_unique
assert dates.is_monotonic_increasing

print("Branch:", branch)
print("Number of epochs:", len(dates))
print("First epoch:", dates.iloc[0])
print("Last epoch:", dates.iloc[-1])
print("Output:", output_file.name)

Branch: prelim-experiments
Number of epochs: 8328
First epoch: 2025-01-01 00:00:00
Last epoch: 2025-12-27 23:00:00
Output: lageos1_hujsak_nominal_cx_2025.csv


We will now use Brent's in-built propagation functionality to set up the code to propagate and save the satellite states for both the modified and unmodified SGP4 implementations. Note that this will require running this notebook in `sgp4-baseline` and `prelim-experiments` branch separately.

In [3]:
# Prevent accidental overwriting:
if output_file.exists():
    raise FileExistsError(f"File already exists: {output_file}")
if not tle_path.exists():
    raise FileNotFoundError(f"TLE data not found: {tle_path}")

# Ensure SGP4 implementation has no uncommitted changes:
source_status = subprocess.check_output(
    ["git", "status", "--porcelain", "--", "brent/propagators/vallado_sgp4/valladopy/astro/sgp4/"],
    cwd=repo_path, text=True).strip()

if source_status:
    raise RuntimeError("Commit SGP4 source changes before running.")

# Propagate LAGEOS-1 satellite states:
tle = TLEPropagator.load(path=str(tle_path), method=TLEPropagatorMethod.FORWARD)  # load TLEs
states = []
for timestamp in dates:  # propagate at SP3 epochs
    state = np.asarray(tle._propagate(timestamp.to_pydatetime(), Constants.DEFAULT_ECI),dtype=float)
    states.append(state)
states = np.asarray(states)
assert states.shape == (len(dates), 6)
assert np.isfinite(states).all()

states_df = pd.DataFrame(states,
    columns=["x_m", "y_m", "z_m", "vx_m_s", "vy_m_s", "vz_m_s"])
states_df.insert(0, "epoch_utc", dates)

# Record code version + input data for verification:
states_df["git_commit"] = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_path,
    text=True
).strip()


# Save without overwriting:
states_df.to_csv(output_file, index=False, mode="x")
print("Saved:", output_file)
print("Number of states:", len(states_df))
states_df.head(5)


Saved: C:\Users\lette\OneDrive\Desktop\Imperial PhD\Year 1\brent\data\sgp4mods-comparison\lageos1_hujsak_nominal_cx_2025.csv
Number of states: 8328


,epoch_utc,x_m,y_m,z_m,vx_m_s,vy_m_s,vz_m_s,git_commit
0,2025-01-01 00:00:00,7.603157e+06,-3.635582e+06,8.838233e+06,-4473.423474,-981.607915,3437.361323,a2924936aa74dbe9d38cb833e79b0af1367ed02d
1,2025-01-01 01:00:00,-1.037330e+07,-1.687300e+06,6.330455e+06,-3052.380979,1790.810599,-4469.064601,a2924936aa74dbe9d38cb833e79b0af1367ed02d
2,2025-01-01 02:00:00,-5.577960e+06,4.020308e+06,-1.023652e+07,5048.768311,614.797905,-2507.462828,a2924936aa74dbe9d38cb833e79b0af1367ed02d
3,2025-01-01 03:00:00,1.137949e+07,9.626054e+05,-4.470440e+06,2079.110085,-1909.195622,4955.711593,a2924936aa74dbe9d38cb833e79b0af1367ed02d
4,2025-01-01 04:00:00,3.198053e+06,-4.166657e+06,1.102564e+07,-5515.285484,-233.520243,1518.665033,a2924936aa74dbe9d38cb833e79b0af1367ed02d
